# 01. 올리브영 상품·리뷰 크롤링

**목적**  
크림 상품 목록, 전성분, 피부타입·피부톤별 리뷰를 수집합니다.

**입력**  
`올리브영 크림 카테고리와 상품 상세 페이지`

**출력**  
`data/raw의 상품·전성분 및 리뷰 파일`

> 기본값에서는 실시간 크롤링을 실행하지 않고 저장된 원본 데이터를 확인합니다.


In [ ]:
from pathlib import Path

# Jupyter와 Colab 모두 저장소 루트에서 실행합니다.
def find_project_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "data").is_dir() and (path / "notebooks").is_dir():
            return path
    raise FileNotFoundError("저장소를 clone한 뒤 해당 폴더 안에서 실행하세요.")

PROJECT_ROOT = find_project_root()

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

for directory in [DATA_RAW_DIR, DATA_INTERIM_DIR, DATA_PROCESSED_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

import os

# 안전 기본값: Run all을 눌러도 실시간 사이트에 접속하지 않습니다.
RUN_LIVE_CRAWLING = False
RUN_EXPLORATORY_STEPS = False

PRODUCT_INGREDIENT_PATH = DATA_RAW_DIR / "올리브영_크림240개_상품명_전성분.csv"
REVIEW_OUTPUT_PATH = DATA_RAW_DIR / "올리브영_크림240개_피부타입_피부톤_리뷰.csv"

if RUN_LIVE_CRAWLING:
    DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
    print("라이브 크롤링 모드입니다. 결과는", DATA_RAW_DIR, "에 저장됩니다.")
else:
    print("라이브 크롤링 비활성화: 저장된 원본 데이터를 확인합니다.")


## 수집 코드

아래 코드는 `RUN_LIVE_CRAWLING=True`로 명시한 경우에만 실행됩니다.


In [3]:
if RUN_LIVE_CRAWLING:
    pass
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.chrome.options import Options
    import re
    import time

    # 크롬드라이버 경로
    path = os.getenv("CHROMEDRIVER_PATH", "").strip()

    # 옵션
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")

    # 서비스 연결
    s = Service(path) if path else Service()

    # 드라이버 실행
    driver = webdriver.Chrome(service=s, options=options)

    # 크림 카테고리
    url = "https://www.oliveyoung.co.kr/store/display/getMCategoryList.do?dispCatNo=1000001000100150001&rowsPerPage=48&prdSort=03"


    driver.get(url)


### 1. 크림 상품 목록 수집

크림 카테고리의 판매 목록에서 상품번호, 상품명과 상세 페이지 주소를 수집합니다.


In [ ]:
if RUN_LIVE_CRAWLING:
    pass
    import pandas as pd
    import time
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC

    all_data = []

    rank = 1

    for page in range(1, 11):

        url = (
            "https://www.oliveyoung.co.kr/store/display/getMCategoryList.do"
            f"?dispCatNo=1000001000100150001"
            f"&rowsPerPage=48"
            f"&prdSort=03"
            f"&pageIdx={page}"
        )

        print(f"\n===== {page} 페이지 =====")

        driver.get(url)

        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "button.btn_zzim")
            )
        )

        time.sleep(1)

        buttons = driver.find_elements(
            By.CSS_SELECTOR,
            "button.btn_zzim"
        )

        print("상품 수:", len(buttons))

        for b in buttons:

            all_data.append({
                "순위": rank,
                "페이지": page,
                "상품번호": b.get_attribute("data-ref-goodsno"),
                "상품명": b.get_attribute("data-ref-goodsnm")
            })

            rank += 1

    df = pd.DataFrame(all_data)

    # 중복 제거 (필요하면)
    df = df.drop_duplicates(subset="상품번호")

    # 순위 기준 정렬
    df = df.sort_values("순위").reset_index(drop=True)

    print(df.head())


In [4]:
if RUN_LIVE_CRAWLING:
    print("총 상품 수:", len(df))

    df.to_csv("올리브영_상품목록_상위480개.csv", index=False, encoding="utf-8-sig")



===== 1 페이지 =====
상품 수: 48

===== 2 페이지 =====
상품 수: 48

===== 3 페이지 =====
상품 수: 48

===== 4 페이지 =====
상품 수: 48

===== 5 페이지 =====
상품 수: 48

===== 6 페이지 =====
상품 수: 48

===== 7 페이지 =====
상품 수: 48

===== 8 페이지 =====
상품 수: 48

===== 9 페이지 =====
상품 수: 48

===== 10 페이지 =====
상품 수: 48
   순위  페이지           상품번호                                                상품명
0   1    1  A000000260257      [7/21하루특가][대용량 기획] 제로이드 수딩 크림 80ml 기획 (+50ml)
1   2    1  A000000226553  [수부지크림/유수분밸런스] 브링그린 티트리 시카 수딩 크림 플러스 100ml (2입...
2   3    1  A000000192782   [1+1/1등 모공 수분천재크림] 에스네이처 아쿠아 스쿠알란 수분크림 60ml 더블기획
3   4    1  A000000233337                  [수분진정/광채캡슐크림] 션리 다시마 글레이즈드 크림 50g
4   5    1  A000000253592    [흔적,기미,미백/TXA] 셀리맥스 트라넥삼산 잡티 크림 35ml 기획 (+14ml)
총 상품 수: 480


In [ ]:
if RUN_LIVE_CRAWLING:
    pass
    import pandas as pd
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    import time

    # 크롬드라이버 경로
    path = os.getenv("CHROMEDRIVER_PATH", "").strip()

    # 옵션
    options = Options()

    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")

    # 서비스 연결
    s = Service(path) if path else Service()

    # 드라이버 실행
    driver = webdriver.Chrome(
        service=s,
        options=options
    )

    # 1. '주성분' 컬럼이 없다면 생성
    if '주성분' not in df.columns:
        df['주성분'] = None

    # 2. 전체 루프 시작


### 2. 상품별 전성분 수집

각 상품 상세 페이지에서 전성분 문자열을 가져와 상품 정보와 연결합니다.


In [ ]:
if RUN_LIVE_CRAWLING:
    for index, row in df.iterrows():
        goods_no = row['상품번호']

        # 이미 수집된 데이터는 건너뜁니다 (코드 재실행 시 매우 유용)
        if pd.notnull(df.loc[index, '주성분']) and df.loc[index, '주성분'] != "수집실패":
            continue

        print(f"[{index+1}/{len(df)}] 수집 중: {goods_no}")

        url = f"https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo={goods_no}"
        driver.get(url)
        time.sleep(3) # 페이지 안정적 로딩 대기

        try:
            # 탭 클릭 (상품정보 제공고시)
            tab_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="tab-panels"]/section/ul/li[1]/button/span'))
            )
            tab_button.click()
            time.sleep(1.5) # 클릭 후 테이블 로딩 대기

            # 성분 추출 (tr[8]/td 경로 사용)
            element = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.XPATH, '//*[@id="tab-panels"]/section/ul/li[1]/div/div/table/tbody/tr[8]/td'))
            )

            # df에 즉시 반영
            extracted_text = element.text
            df.loc[index, '주성분'] = extracted_text
            print(f"  > 성공: {extracted_text[:20]}...")

        except Exception as e:
            df.loc[index, '주성분'] = "수집실패"
            print(f"  > 실패: {goods_no}")

        # 3. 20개마다 중간 저장 (CSV 파일로 데이터 보호)
        if (index + 1) % 20 == 0:
            df.to_csv('올리브영_상품정보_업데이트.csv', index=False, encoding='utf-8-sig')
            print(">> 20개 단위 중간 저장 완료")

    # 4. 최종 저장
    df.to_csv('올리브영_크림_상품정보_최종.csv', index=False, encoding='utf-8-sig')
    driver.quit()
    print("\n--- 전체 수집 완료! ---")


[1/480] 수집 중: A000000260257
  > 성공: 정제수, 글리세린, 프로판다이올, 카...
[2/480] 수집 중: A000000226553
  > 성공: 정제수, 부틸렌글라이콜, 스쿠알란, ...
[3/480] 수집 중: A000000192782
  > 성공: 정제수, 스쿠알란(150,000ppm...
[4/480] 수집 중: A000000233337
  > 성공: 정제수, 다시마즙 (20%), 사이클...
[5/480] 수집 중: A000000253592
  > 성공: 전성분 정제수, 글리세린, 나이아신아...
[6/480] 수집 중: A000000222833
  > 성공: 정제수, 부틸렌글라이콜, 글리세린, ...
[7/480] 수집 중: A000000258834
  > 성공: 정제수, 글리세린, 부틸렌글라이콜, ...
[8/480] 수집 중: A000000258214
  > 성공: 정제수, 부틸렌글라이콜, 1,2-헥산...
[19/480] 수집 중: A000000246297
  > 성공: 정제수, 글리세린, 하이드로제네이티드...
[20/480] 수집 중: A000000224581
  > 성공: [본품/증정 동일 - 프로즌 크림] ...
>> 20개 단위 중간 저장 완료
[21/480] 수집 중: A000000258294
  > 성공: 정제수, 부틸렌글라이콜, 글리세린, ...
[22/480] 수집 중: A000000224727
  > 성공: 정제수, 프로판다이올, 글리세린, 글...
[23/480] 수집 중: A000000134691
  > 성공: 정제수, 글리세린, 미네랄오일, 카프...
[24/480] 수집 중: A000000250472
  > 성공: 정제수, 글리세린, 부틸렌글라이콜, ...
[25/480] 수집 중: A000000211213
  > 성공: [모이스처라이징 밤-본품/증정] 정제...
[26/480] 수집 중: A000000222882
  > 성공: 정제수, 글리세린, 부틸렌글라이콜, ...
[27/480] 수집 중

In [1]:
if RUN_LIVE_CRAWLING:
    pass
    import pandas as pd
    df = pd.read_csv("올리브영_크림_상품정보_최종.csv", encoding = 'utf-8-sig')


In [5]:
if RUN_LIVE_CRAWLING:
    pass
    df_new = df[:240]


In [46]:
if RUN_EXPLORATORY_STEPS:
    pass
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.chrome.options import Options
    import re
    import time

    # 크롬드라이버 경로
    path = os.getenv("CHROMEDRIVER_PATH", "").strip()

    # 옵션
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")

    # 서비스 연결
    s = Service(path) if path else Service()

    # 드라이버 실행
    driver = webdriver.Chrome(service=s, options=options)

    url = "https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000250781&tab=review"


    driver.get(url)


In [17]:
if RUN_LIVE_CRAWLING:
    pass
    # 지성 :A01, 건성: A02, 복합성:A03, 민감성:A04, 약건성:A05, 트러블성:A06, 중성:A07
    # 쿨톤: B01, 웜톤: B02, 봄웜톤:B03, 여름쿨톤:B04, 가을웜톤:B05, 겨울쿨톤:B06
    # 평점 높은 순 RATING_DESC, 평점 낮은 순 RATING_ASC, 유용한 순 USEFUL_SCORE_DESC


In [ ]:
if RUN_EXPLORATORY_STEPS:
    pass
    import pandas as pd
    import time
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.chrome.options import Options

    # -----------------------
    # 크롬 실행
    # -----------------------
    path = os.getenv("CHROMEDRIVER_PATH", "").strip()

    driver = webdriver.Chrome(
        service=Service(path) if path else Service(),
        options=Options()
    )

    # -----------------------
    # 상품 상세페이지 접속
    # -----------------------
    driver.get(
        "https://m.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000260257&tab=review"
    )

    time.sleep(3)

    # -----------------------
    # 테스트할 상품번호
    # -----------------------
    goods_no = "A000000260257"

    # -----------------------
    # API 호출
    # -----------------------
    script = f"""
    return fetch(
        "https://m.oliveyoung.co.kr/review/api/v2/reviews/cursor",
        {{
            method: "POST",
            headers: {{
                "Content-Type": "application/json"
            }},
            body: JSON.stringify({{
                goodsNumber: "{goods_no}",
                page: 0,
                size: 20,
                sortType: "RATING_DESC",
                reviewType: "ALL",
                skinType: "A01"
            }})
        }}
    ).then(res => res.json());
    """

    result = driver.execute_script(script)

    # -----------------------
    # 확인
    # -----------------------
    reviews = result["data"]["goodsReviewList"]


In [8]:
if RUN_EXPLORATORY_STEPS:
    print("리뷰 개수:", len(reviews))

    for i, review in enumerate(reviews, start=1):

        print(f"\n===== 리뷰 {i} =====")

        print("평점 :", review.get("reviewScore"))
        print("내용 :", review.get("content"))

    driver.quit()


리뷰 개수: 20

===== 리뷰 1 =====
평점 : 5
내용 : 제로이드 크림 유명해서 사봤는데 이유가 있네용 보습 잘됨 .

===== 리뷰 2 =====
평점 : 5
내용 : 진짜 자극 없이 순해서 아무때나 바르기 좋은 것 같아요 순해서 뭐 딱히 나는것도 없어서 계속 구매할 것 같아요

===== 리뷰 3 =====
평점 : 5
내용 : 다른 크림들은 바른 후애 번들번들 올라와서 싫었는데 제로이드 수딩은 그런 느낌 없어 바로 흡수돼서 피부가 보송해져요

===== 리뷰 4 =====
평점 : 5
내용 : 리뷰가 좋아서 구매했었는데 끈적이지않고 바를 수 잌ㅅ어서 좋아ㅛ요

===== 리뷰 5 =====
평점 : 5
내용 : 너무 기름지지도 않고 너무 가볍지도않고 그래서 쓰기 좋음 지성인데 좀 무거운거 쓰고싶으면 추천

===== 리뷰 6 =====
평점 : 5
내용 : 여름엔 지성피부라 쓰기 좀 그렇고 겨울엔 이만한 보습 크림 없는 것 같어요

===== 리뷰 7 =====
평점 : 5
내용 : 자극없이순하고 데이케어용 보습으로 사용하기 좋아요. 화장 전에도 나름 괜찮은 것 같아요

===== 리뷰 8 =====
평점 : 5
내용 : 먼저 제 피부 타입을 설명드리자면 좁쌀, 화농성, 염증성 여드름과 가끔씩 모낭염이 생기는 속건조 살짝 있는 지성 피부입니다. 

일단 지성 피부는 기름을 닦아내기 위해서 과도하게 세정하거나 하는 경우로 장벽이 깨지는 경우가 너무 많은데 장벽을 개선하자니 유명한 장벽크림들은 너무 기름져서 지성들에겐 화농성이 생길 수 밖에 없겠더라구요.. 이건 전혀 그런 것도 없고 오히려 여드름, 모낭염, 각종 트러블이 싹 사라졌어요!!! 피부 컨디션 자체가 올라간게 느껴졌고 그냥 개미쳤습니다!!! 물론 아무래도 크림이라보니 조금은 기름질 수 있으나 얇게 발라주면 문제 없다 생각합니다.

제형은 진짜 그냥 좀 되직한 수분감 있는 크림 정도고요 흡수 잘 됩니다. 발라도 답답하다는 느낌이 적어서 너무 좋았습니다

참고로 저는 수분 앰플 2~3번

### 3. 리뷰 수집 설정

피부유형과 피부톤 조건별 요청값, 저장 위치와 실행 범위를 설정합니다.


In [ ]:
if RUN_LIVE_CRAWLING:
    pass
    import os
    import json
    import time
    import random
    import pandas as pd

    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.chrome.options import Options
    from selenium.common.exceptions import (
        WebDriverException,
        TimeoutException,
        JavascriptException,
        InvalidSessionIdException
    )


    # =========================================================
    # 1. 기본 설정
    # =========================================================

    CHROMEDRIVER_PATH = os.getenv("CHROMEDRIVER_PATH", "")

    BASE_URL = (
        "https://m.oliveyoung.co.kr/store/goods/"
        "getGoodsDetail.do?goodsNo=A000000260257"
    )

    OUTPUT_FILE = REVIEW_OUTPUT_PATH
    CHECKPOINT_FILE = DATA_INTERIM_DIR / "올리브영_리뷰수집_체크포인트.csv"
    PROGRESS_FILE = DATA_INTERIM_DIR / "oliveyoung_reviews_progress.json"
    ERROR_FILE = DATA_INTERIM_DIR / "올리브영_리뷰수집_오류목록.csv"

    # 한 번의 API 요청에서 가져오는 리뷰 수
    PAGE_SIZE = 10

    # 요청 실패 시 최대 재시도 횟수
    MAX_RETRIES = 3

    # fetch 요청 제한시간
    FETCH_TIMEOUT_MS = 20000

    # 정상 요청 사이 대기시간
    MIN_REQUEST_DELAY = 0.2
    MAX_REQUEST_DELAY = 0.5

    # 상품 하나가 끝난 뒤 대기시간
    MIN_PRODUCT_DELAY = 0.5
    MAX_PRODUCT_DELAY = 1.0


    # =========================================================
    # 2. 피부타입
    # =========================================================


In [ ]:
if RUN_LIVE_CRAWLING:
    skin_types = {
        "A01": "지성",
        "A02": "건성",
        "A03": "복합성",
        "A04": "민감성",
        "A05": "약건성",
        "A06": "트러블성",
        "A07": "중성"
    }


    # =========================================================
    # 3. 피부톤
    # =========================================================

    skin_tones = {
        "B01": "쿨톤",
        "B02": "웜톤",
        "B03": "봄웜톤",
        "B04": "여름쿨톤",
        "B05": "가을웜톤",
        "B06": "겨울쿨톤"
    }


    # =========================================================
    # 4. 정렬 방식
    # =========================================================

    sort_types = {
        "USEFUL_SCORE_DESC": "유용한순"

        # 필요하면 아래 정렬도 추가
        # "RATING_DESC": "평점높은순",
        # "RATING_ASC": "평점낮은순"
    }


    # =========================================================
    # 5. ChromeDriver 생성
    # =========================================================


### 4. 브라우저 및 저장 함수

Selenium 브라우저를 준비하고 중간 수집 결과를 안전하게 이어서 저장할 수 있도록 구성합니다.


In [ ]:
if RUN_LIVE_CRAWLING:
    def create_driver():
        options = Options()

        options.add_argument("--start-maximized")
        options.add_argument(
            "--disable-blink-features=AutomationControlled"
        )

        options.add_argument("--disable-gpu")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--no-sandbox")

        options.add_experimental_option(
            "excludeSwitches",
            ["enable-logging", "enable-automation"]
        )

        options.add_experimental_option(
            "useAutomationExtension",
            False
        )

        driver = webdriver.Chrome(
            service=(Service(CHROMEDRIVER_PATH) if CHROMEDRIVER_PATH else Service()),
            options=options
        )

        # execute_async_script 최대 대기시간
        driver.set_script_timeout(30)

        driver.get(BASE_URL)
        time.sleep(2)

        print("ChromeDriver 실행 완료")

        return driver


    # =========================================================
    # 6. ChromeDriver 상태 확인
    # =========================================================

    def is_driver_alive(driver):
        if driver is None:
            return False

        try:
            driver.current_url
            return True

        except Exception:
            return False


    # =========================================================
    # 7. ChromeDriver 재시작
    # =========================================================


In [ ]:
if RUN_LIVE_CRAWLING:
    def restart_driver(driver):
        print("ChromeDriver를 다시 실행합니다.")

        try:
            if driver is not None:
                driver.quit()

        except Exception:
            pass

        time.sleep(3)

        return create_driver()


    # =========================================================
    # 8. 완료된 요청 불러오기
    # =========================================================

    def load_progress():
        if not os.path.exists(PROGRESS_FILE):
            return set()

        try:
            with open(
                PROGRESS_FILE,
                "r",
                encoding="utf-8"
            ) as file:
                progress_data = json.load(file)

            return set(progress_data)

        except Exception as error:
            print("진행 상황 파일 불러오기 실패:", error)
            return set()


    # =========================================================
    # 9. 완료된 요청 저장
    # =========================================================


In [ ]:
if RUN_LIVE_CRAWLING:
    def save_progress(completed_requests):
        temp_file = PROGRESS_FILE + ".tmp"

        try:
            with open(
                temp_file,
                "w",
                encoding="utf-8"
            ) as file:
                json.dump(
                    sorted(completed_requests),
                    file,
                    ensure_ascii=False,
                    indent=2
                )

            os.replace(temp_file, PROGRESS_FILE)

        except Exception as error:
            print("진행 상황 저장 실패:", error)


    # =========================================================
    # 10. 기존 중간 저장 데이터 불러오기
    # =========================================================

    def load_existing_reviews():
        if not os.path.exists(CHECKPOINT_FILE):
            return []

        try:
            checkpoint_df = pd.read_csv(
                CHECKPOINT_FILE,
                dtype={
                    "상품번호": str,
                    "피부타입코드": str,
                    "피부톤코드": str
                },
                encoding="utf-8-sig"
            )

            print(
                f"기존 중간 저장 데이터 "
                f"{len(checkpoint_df):,}개를 불러왔습니다."
            )

            return checkpoint_df.to_dict("records")

        except Exception as error:
            print("중간 저장 데이터 불러오기 실패:", error)
            return []


    # =========================================================
    # 11. 중복 제거
    # =========================================================


In [ ]:
if RUN_LIVE_CRAWLING:
    def remove_duplicates(review_df):
        if review_df.empty:
            return review_df

        # 리뷰번호가 존재하면 리뷰번호 기준으로 중복 제거
        if (
            "리뷰번호" in review_df.columns
            and review_df["리뷰번호"].notna().any()
        ):
            subset_columns = [
                "상품번호",
                "피부타입코드",
                "피부톤코드",
                "리뷰번호"
            ]

        # 리뷰번호가 없으면 리뷰 내용 기준으로 중복 제거
        else:
            subset_columns = [
                "상품번호",
                "피부타입코드",
                "피부톤코드",
                "평점",
                "리뷰내용"
            ]

        subset_columns = [
            column
            for column in subset_columns
            if column in review_df.columns
        ]

        review_df = review_df.drop_duplicates(
            subset=subset_columns
        )

        return review_df.reset_index(drop=True)


    # =========================================================
    # 12. 중간 저장
    # =========================================================


In [ ]:
if RUN_LIVE_CRAWLING:
    def save_checkpoint(all_reviews):
        review_df = pd.DataFrame(all_reviews)

        if review_df.empty:
            return review_df

        review_df = remove_duplicates(review_df)

        temp_file = CHECKPOINT_FILE + ".tmp"

        review_df.to_csv(
            temp_file,
            index=False,
            encoding="utf-8-sig"
        )

        os.replace(temp_file, CHECKPOINT_FILE)

        return review_df


    # =========================================================
    # 13. 오류 기록 저장
    # =========================================================

    def save_errors(error_records):
        if not error_records:
            return

        error_df = pd.DataFrame(error_records)

        error_df.to_csv(
            ERROR_FILE,
            index=False,
            encoding="utf-8-sig"
        )


    # =========================================================
    # 14. Olive Young API 요청
    # =========================================================


### 5. 리뷰 요청 및 응답 변환

리뷰 응답에서 분석에 필요한 상품·피부유형·피부톤·리뷰 정보를 표 형태로 변환합니다.


In [ ]:
if RUN_LIVE_CRAWLING:
    def fetch_reviews(
        driver,
        goods_no,
        skin_code,
        tone_code,
        sort_code
    ):
        script = """
        const goodsNumber = arguments[0];
        const skinType = arguments[1];
        const skinTone = arguments[2];
        const sortType = arguments[3];
        const pageSize = arguments[4];
        const timeoutMs = arguments[5];

        const callback = arguments[arguments.length - 1];

        const controller = new AbortController();

        const timeoutId = setTimeout(() => {
            controller.abort();
        }, timeoutMs);

        fetch(
            "https://m.oliveyoung.co.kr/review/api/v2/reviews/cursor",
            {
                method: "POST",

                headers: {
                    "Content-Type": "application/json",
                    "Accept": "application/json"
                },

                credentials: "include",

                signal: controller.signal,

                body: JSON.stringify({
                    goodsNumber: goodsNumber,
                    page: 0,
                    size: pageSize,
                    sortType: sortType,
                    reviewType: "ALL",
                    skinType: skinType,
                    skinTone: skinTone
                })
            }
        )
        .then(async response => {
            const text = await response.text();

            clearTimeout(timeoutId);

            if (!response.ok) {
                callback({
                    success: false,
                    status: response.status,
                    message: text
                });

                return;
            }

            try {
                callback({
                    success: true,
                    status: response.status,
                    data: JSON.parse(text)
                });

            } catch (error) {
                callback({
                    success: false,
                    status: response.status,
                    message: "JSON 파싱 오류: " + error.message
                });
            }
        })
        .catch(error => {
            clearTimeout(timeoutId);

            callback({
                success: false,
                status: 0,
                message: error.name + ": " + error.message
            });
        });
        """

        return driver.execute_async_script(
            script,
            str(goods_no),
            skin_code,
            tone_code,
            sort_code,
            PAGE_SIZE,
            FETCH_TIMEOUT_MS
        )


    # =========================================================
    # 15. 리뷰 데이터 정리
    # =========================================================


In [ ]:
if RUN_LIVE_CRAWLING:
    def parse_reviews(
        reviews,
        goods_no,
        skin_code,
        skin_name,
        tone_code,
        tone_name,
        sort_code,
        sort_name
    ):
        parsed_reviews = []

        for review in reviews:
            parsed_reviews.append({
                "상품번호": str(goods_no),

                "피부타입코드": skin_code,
                "피부타입": skin_name,

                "피부톤코드": tone_code,
                "피부톤": tone_name,

                "정렬코드": sort_code,
                "정렬": sort_name,

                "리뷰번호": (
                    review.get("goodsReviewNo")
                    or review.get("reviewNo")
                    or review.get("id")
                ),

                "평점": review.get("reviewScore"),
                "리뷰내용": review.get("content"),

                "작성일": (
                    review.get("createdDate")
                    or review.get("createDate")
                    or review.get("reviewDate")
                ),

                "도움수": (
                    review.get("usefulScore")
                    or review.get("helpCount")
                )
            })

        return parsed_reviews


    # =========================================================
    # 16. 메인 수집 함수
    # =========================================================


### 6. 전체 리뷰 수집 함수

상품과 피부 조건을 순회하며 중복을 확인하고 수집 결과를 누적합니다.


In [ ]:
if RUN_LIVE_CRAWLING:
    def get_goods_list(product_df):
        if "상품번호" not in product_df.columns:
            raise KeyError("상품 데이터에 '상품번호' 열이 없습니다.")

        goods_list = (
            product_df["상품번호"]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )
        return [
            goods_no
            for goods_no in goods_list
            if goods_no and goods_no.lower() != "nan"
        ]


In [ ]:
if RUN_LIVE_CRAWLING:
    def request_review_combination(
        driver,
        goods_no,
        skin_code,
        skin_name,
        tone_code,
        tone_name,
        sort_code,
        sort_name,
    ):
        last_error_message = ""

        for retry in range(1, MAX_RETRIES + 1):
            try:
                if not is_driver_alive(driver):
                    driver = restart_driver(driver)

                result = fetch_reviews(
                    driver=driver,
                    goods_no=goods_no,
                    skin_code=skin_code,
                    tone_code=tone_code,
                    sort_code=sort_code,
                )
                if not isinstance(result, dict):
                    raise ValueError(f"잘못된 응답: {result}")

                status = result.get("status", 0)
                if not result.get("success", False):
                    message = result.get("message", "알 수 없는 오류")
                    last_error_message = f"HTTP {status}: {message}"
                    print(
                        f"[실패 {retry}/{MAX_RETRIES}] "
                        f"{skin_name} | {tone_name} | {last_error_message}"
                    )
                    if status in {403, 429}:
                        wait_time = 10 * retry + random.uniform(1, 3)
                    elif status in {500, 502, 503, 504}:
                        wait_time = 5 * retry + random.uniform(1, 2)
                    else:
                        wait_time = 2 * retry
                    time.sleep(wait_time)
                    continue

                response_data = result.get("data", {})
                reviews = response_data.get("data", {}).get("goodsReviewList", [])
                if not reviews:
                    reviews = response_data.get("goodsReviewList", [])

                parsed_reviews = parse_reviews(
                    reviews=reviews or [],
                    goods_no=goods_no,
                    skin_code=skin_code,
                    skin_name=skin_name,
                    tone_code=tone_code,
                    tone_name=tone_name,
                    sort_code=sort_code,
                    sort_name=sort_name,
                )
                time.sleep(random.uniform(MIN_REQUEST_DELAY, MAX_REQUEST_DELAY))
                return driver, parsed_reviews, ""

            except (InvalidSessionIdException, WebDriverException) as error:
                last_error_message = f"{type(error).__name__}: {error}"
                print(f"[Chrome 오류 {retry}/{MAX_RETRIES}]")
                driver = restart_driver(driver)
            except TimeoutException as error:
                last_error_message = f"TimeoutException: {error}"
                print(f"[시간 초과 {retry}/{MAX_RETRIES}] {skin_name} | {tone_name}")
                time.sleep(2 * retry)
            except JavascriptException as error:
                last_error_message = f"JavascriptException: {error}"
                print(f"[JavaScript 오류 {retry}/{MAX_RETRIES}]")
                time.sleep(2 * retry)
            except Exception as error:
                last_error_message = f"{type(error).__name__}: {error}"
                print(f"[기타 오류 {retry}/{MAX_RETRIES}] {last_error_message}")
                time.sleep(2 * retry)

        return driver, None, last_error_message


In [ ]:
if RUN_LIVE_CRAWLING:
    def collect_product_reviews(
        driver,
        goods_no,
        completed_requests,
        all_reviews,
        error_records,
    ):
        collected_count = 0
        failed_count = 0

        for skin_code, skin_name in skin_types.items():
            for tone_code, tone_name in skin_tones.items():
                for sort_code, sort_name in sort_types.items():
                    request_key = f"{goods_no}|{skin_code}|{tone_code}|{sort_code}"
                    if request_key in completed_requests:
                        continue

                    driver, parsed_reviews, error_message = request_review_combination(
                        driver,
                        goods_no,
                        skin_code,
                        skin_name,
                        tone_code,
                        tone_name,
                        sort_code,
                        sort_name,
                    )
                    if parsed_reviews is None:
                        failed_count += 1
                        error_records.append({
                            "상품번호": goods_no,
                            "피부타입코드": skin_code,
                            "피부타입": skin_name,
                            "피부톤코드": tone_code,
                            "피부톤": tone_name,
                            "정렬코드": sort_code,
                            "정렬": sort_name,
                            "오류내용": error_message,
                            "발생시간": pd.Timestamp.now(),
                        })
                        continue

                    all_reviews.extend(parsed_reviews)
                    completed_requests.add(request_key)
                    collected_count += len(parsed_reviews)
                    print(f"{skin_name:<4} | {tone_name:<5} | {len(parsed_reviews):>2}개")

        return driver, collected_count, failed_count


In [ ]:
if RUN_LIVE_CRAWLING:
    def get_goods_list(product_df):
        if "상품번호" not in product_df.columns:
            raise KeyError("상품 데이터에 '상품번호' 열이 없습니다.")

        goods_list = (
            product_df["상품번호"]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )
        return [
            goods_no
            for goods_no in goods_list
            if goods_no and goods_no.lower() != "nan"
        ]


In [ ]:
if RUN_LIVE_CRAWLING:
    def request_review_combination(
        driver,
        goods_no,
        skin_code,
        skin_name,
        tone_code,
        tone_name,
        sort_code,
        sort_name,
    ):
        last_error_message = ""

        for retry in range(1, MAX_RETRIES + 1):
            try:
                if not is_driver_alive(driver):
                    driver = restart_driver(driver)

                result = fetch_reviews(
                    driver=driver,
                    goods_no=goods_no,
                    skin_code=skin_code,
                    tone_code=tone_code,
                    sort_code=sort_code,
                )
                if not isinstance(result, dict):
                    raise ValueError(f"잘못된 응답: {result}")

                status = result.get("status", 0)
                if not result.get("success", False):
                    message = result.get("message", "알 수 없는 오류")
                    last_error_message = f"HTTP {status}: {message}"
                    print(
                        f"[실패 {retry}/{MAX_RETRIES}] "
                        f"{skin_name} | {tone_name} | {last_error_message}"
                    )
                    if status in {403, 429}:
                        wait_time = 10 * retry + random.uniform(1, 3)
                    elif status in {500, 502, 503, 504}:
                        wait_time = 5 * retry + random.uniform(1, 2)
                    else:
                        wait_time = 2 * retry
                    time.sleep(wait_time)
                    continue

                response_data = result.get("data", {})
                reviews = response_data.get("data", {}).get("goodsReviewList", [])
                if not reviews:
                    reviews = response_data.get("goodsReviewList", [])

                parsed_reviews = parse_reviews(
                    reviews=reviews or [],
                    goods_no=goods_no,
                    skin_code=skin_code,
                    skin_name=skin_name,
                    tone_code=tone_code,
                    tone_name=tone_name,
                    sort_code=sort_code,
                    sort_name=sort_name,
                )
                time.sleep(random.uniform(MIN_REQUEST_DELAY, MAX_REQUEST_DELAY))
                return driver, parsed_reviews, ""

            except (InvalidSessionIdException, WebDriverException) as error:
                last_error_message = f"{type(error).__name__}: {error}"
                print(f"[Chrome 오류 {retry}/{MAX_RETRIES}]")
                driver = restart_driver(driver)
            except TimeoutException as error:
                last_error_message = f"TimeoutException: {error}"
                print(f"[시간 초과 {retry}/{MAX_RETRIES}] {skin_name} | {tone_name}")
                time.sleep(2 * retry)
            except JavascriptException as error:
                last_error_message = f"JavascriptException: {error}"
                print(f"[JavaScript 오류 {retry}/{MAX_RETRIES}]")
                time.sleep(2 * retry)
            except Exception as error:
                last_error_message = f"{type(error).__name__}: {error}"
                print(f"[기타 오류 {retry}/{MAX_RETRIES}] {last_error_message}")
                time.sleep(2 * retry)

        return driver, None, last_error_message


In [ ]:
if RUN_LIVE_CRAWLING:
    def collect_product_reviews(
        driver,
        goods_no,
        completed_requests,
        all_reviews,
        error_records,
    ):
        collected_count = 0
        failed_count = 0

        for skin_code, skin_name in skin_types.items():
            for tone_code, tone_name in skin_tones.items():
                for sort_code, sort_name in sort_types.items():
                    request_key = f"{goods_no}|{skin_code}|{tone_code}|{sort_code}"
                    if request_key in completed_requests:
                        continue

                    driver, parsed_reviews, error_message = request_review_combination(
                        driver,
                        goods_no,
                        skin_code,
                        skin_name,
                        tone_code,
                        tone_name,
                        sort_code,
                        sort_name,
                    )
                    if parsed_reviews is None:
                        failed_count += 1
                        error_records.append({
                            "상품번호": goods_no,
                            "피부타입코드": skin_code,
                            "피부타입": skin_name,
                            "피부톤코드": tone_code,
                            "피부톤": tone_name,
                            "정렬코드": sort_code,
                            "정렬": sort_name,
                            "오류내용": error_message,
                            "발생시간": pd.Timestamp.now(),
                        })
                        continue

                    all_reviews.extend(parsed_reviews)
                    completed_requests.add(request_key)
                    collected_count += len(parsed_reviews)
                    print(f"{skin_name:<4} | {tone_name:<5} | {len(parsed_reviews):>2}개")

        return driver, collected_count, failed_count


In [ ]:
if RUN_LIVE_CRAWLING:
    def crawl_reviews(product_df):
        goods_list = get_goods_list(product_df)
        print(f"전체 상품 수: {len(goods_list):,}개")

        all_reviews = load_existing_reviews()
        completed_requests = load_progress()
        error_records = []
        driver = None

        print(f"이미 완료한 요청 조건: {len(completed_requests):,}개")

        try:
            driver = create_driver()
            for product_index, goods_no in enumerate(goods_list, start=1):
                print("\n" + "=" * 65)
                print(f"[{product_index}/{len(goods_list)}] 상품번호: {goods_no}")
                print("=" * 65)
                product_start_time = time.time()

                driver, collected_count, failed_count = collect_product_reviews(
                    driver,
                    goods_no,
                    completed_requests,
                    all_reviews,
                    error_records,
                )

                checkpoint_df = save_checkpoint(all_reviews)
                save_progress(completed_requests)
                save_errors(error_records)
                elapsed_time = time.time() - product_start_time

                print("-" * 65)
                print(
                    f"상품 처리 완료 | 수집 {collected_count:,}개 | "
                    f"실패 조건 {failed_count}개 | {elapsed_time:.1f}초"
                )
                print(f"현재 고유 리뷰 수: {len(checkpoint_df):,}개")

                if product_index < len(goods_list):
                    time.sleep(random.uniform(MIN_PRODUCT_DELAY, MAX_PRODUCT_DELAY))

        except KeyboardInterrupt:
            print("\n사용자가 수집을 중단했습니다. 현재까지 데이터를 저장합니다.")
        except Exception as error:
            print("\n예상하지 못한 오류가 발생했습니다.")
            print(type(error).__name__, ":", error)
        finally:
            save_checkpoint(all_reviews)
            save_progress(completed_requests)
            save_errors(error_records)
            if driver is not None:
                try:
                    driver.quit()
                    print("ChromeDriver 종료 완료")
                except Exception:
                    pass

        final_df = remove_duplicates(pd.DataFrame(all_reviews))
        final_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
        print("\n" + "=" * 65)
        print("수집 종료")
        print("=" * 65)
        print(f"최종 리뷰 수: {len(final_df):,}개")
        print(f"최종 파일: {OUTPUT_FILE}")
        if error_records:
            print(f"실패 요청 수: {len(error_records):,}개")
            print(f"오류 파일: {ERROR_FILE}")
        return final_df


### 7. 리뷰 수집 실행

실시간 수집은 `RUN_LIVE_CRAWLING=True`일 때만 시작됩니다. 기본값에서는 기존 데이터를 변경하지 않습니다.


In [ ]:
if RUN_LIVE_CRAWLING:
    pass
    # 멈출때마다 이 코드 실행하면 저장된 거 불러오고 나머지 실행
    review_df = crawl_reviews(df_new)


In [53]:
if RUN_LIVE_CRAWLING:
    pass
    df_new.to_csv("올리브영_크림240개_상품명_전성분.csv",index=False)


In [1]:
if RUN_LIVE_CRAWLING:
    pass
    import pandas as pd
    df = pd.read_csv("올리브영_크림240개_피부타입_피부톤_리뷰.csv",encoding = "utf-8")


## 저장된 수집 결과

저장소에 포함된 원본 데이터의 크기와 주요 컬럼을 확인합니다.


In [ ]:
import pandas as pd

ingredient_data = pd.read_csv(
    DATA_RAW_DIR / "올리브영_크림240개_상품명_전성분.csv",
    encoding="utf-8-sig",
)
review_data = pd.read_csv(
    DATA_RAW_DIR / "올리브영_크림240개_피부타입_피부톤_리뷰.csv",
    encoding="utf-8-sig",
)

print("상품·성분 데이터:", ingredient_data.shape)
display(ingredient_data.head())
print("리뷰 데이터:", review_data.shape)
display(review_data.head())
